<a href="https://colab.research.google.com/github/eduardmendoza92/Mendoza1706062013Parcial4/blob/main/Parcial4_E_Agrupacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [3]:
url = "https://raw.githubusercontent.com/eduardmendoza92/Mendoza1706062013Parcial4/refs/heads/main/data/raw/clave_E_agrupacion.csv"

df = pd.read_csv(url)
print("Dataset Cargado Correctamente")


Dataset Cargado Correctamente


In [4]:
df.shape

(246, 8)

In [5]:
df.columns

Index(['registro_id', 'edad', 'ingresos', 'frecuencia_uso', 'gasto_promedio',
       'satisfaccion', 'reclamos', 'antiguedad_meses'],
      dtype='object')

In [6]:
df.head()

,registro_id,edad,ingresos,frecuencia_uso,gasto_promedio,satisfaccion,reclamos,antiguedad_meses
0,E-R0096,39,986,7.83,99.00,8.08,2,24
1,E-R0092,41,905,5.52,82.98,8.07,2,23
2,E-R0079,34,1109,7.17,65.25,8.46,0,13
3,E-R0019,32,618,2.57,25.75,5.09,12,21
4,E-R0174,49,1592,8.08,172.04,6.55,0,28


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   registro_id       246 non-null    object 
 1   edad              246 non-null    int64  
 2   ingresos          246 non-null    int64  
 3   frecuencia_uso    246 non-null    float64
 4   gasto_promedio    246 non-null    float64
 5   satisfaccion      245 non-null    float64
 6   reclamos          246 non-null    int64  
 7   antiguedad_meses  246 non-null    int64  
dtypes: float64(3), int64(4), object(1)
memory usage: 15.5+ KB


In [8]:
df.isnull().sum()

,0
registro_id,0
edad,0
ingresos,0
frecuencia_uso,0
gasto_promedio,0
satisfaccion,1
reclamos,0
antiguedad_meses,0


In [9]:
df.duplicated().sum()

np.int64(0)

In [11]:
df['satisfaccion'] = df['satisfaccion'].fillna(df['satisfaccion'].median())

# ==========================================
# 2. Selección de Variables y Normalización
# ==========================================
# Excluimos el identificador 'registro_id'
features = ['edad', 'ingresos', 'frecuencia_uso', 'gasto_promedio', 'satisfaccion', 'reclamos', 'antiguedad_meses']
X = df[features]

# Escalado de datos debido a las diferentes magnitudes (ej. ingresos vs reclamos)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==========================================
# 3. Método del Codo para Selección de K
# ==========================================
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Graficar el método del codo
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--', color='#2c3e50')
plt.title('Método del Codo para Selección de Clústeres')
plt.xlabel('Número de Clústeres (K)')
plt.ylabel('Inercia (WCSS)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('metodo_del_codo.png')
plt.close()

# ==========================================
# 4. Aplicación de K-Means (K = 4 óptimo)
# ==========================================
k_optimo = 4
kmeans_final = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(X_scaled)

# Guardar el dataset transformado con su respectiva etiqueta de grupo
df.to_csv('clave_E_agrupacion_con_clusters.csv', index=False)

# ==========================================
# 5. Visualización de los Grupos
# ==========================================
# Gráfico 1: Relación de valor comercial (Ingresos vs Gasto Promedio)
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x='ingresos', y='gasto_promedio', hue='cluster', palette='Set1', s=100, alpha=0.8)
plt.title('Visualización de Clústeres: Ingresos vs Gasto Promedio')
plt.xlabel('Ingresos Mensuales')
plt.ylabel('Gasto Promedio en el Servicio')
plt.legend(title='Clúster')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig('visualizacion_clusters_ingresos_gasto.png')
plt.close()

# Gráfico 2: Relación de fricción y experiencia (Satisfacción vs Reclamos)
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x='satisfaccion', y='reclamos', hue='cluster', palette='Set1', s=100, alpha=0.8)
plt.title('Visualización de Clústeres: Satisfacción vs Reclamos')
plt.xlabel('Nivel de Satisfacción (0-10)')
plt.ylabel('Cantidad de Reclamos')
plt.legend(title='Clúster')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig('visualizacion_clusters_satisfaccion_reclamos.png')
plt.close()

# ==========================================
# 6. Mostrar Perfil de Medias de los Clústeres
# ==========================================
print("\n--- Promedio de Variables por Clúster ---")
print(df.groupby('cluster')[features].mean().round(2))
print("\n--- Cantidad de clientes por Clúster ---")
print(df['cluster'].value_counts())


--- Promedio de Variables por Clúster ---
          edad  ingresos  frecuencia_uso  gasto_promedio  satisfaccion  \
cluster                                                                  
0        36.12   1020.84            6.19           85.63          8.12   
1        49.67   1628.98            8.90          149.44          8.76   
2        41.41    837.02            3.85           60.61          4.54   
3        26.08    628.62            2.01           32.19          6.07   

         reclamos  antiguedad_meses  
cluster                              
0            1.25             15.99  
1            0.66             35.00  
2            5.24             12.44  
3            3.08              7.19  

--- Cantidad de clientes por Clúster ---
cluster
0    67
3    64
1    61
2    54
Name: count, dtype: int64
